# Sentiment Analysis Submission Notebook (Shabi V2)
Notebook ini mencakup scraping data, preprocessing + labeling, 3 skema pelatihan model, dan inference output kategorikal.

## 0) Setup Path dan Interpreter
Pastikan notebook dijalankan dari folder `shabi_v2/notebooks`.

In [ ]:
import subprocess
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
python_exec = sys.executable

print('Project root:', project_root)
print('Python exe :', python_exec)

## 1) Scraping Data Mandiri (Target 10.000+)
Jika ingin cepat untuk uji awal, ubah `--per-app-target` menjadi `1500` (minimal total 3000).

In [ ]:
scrape_cmd = [
    python_exec,
    str(project_root / 'scripts' / 'scrape_playstore_reviews.py'),
    '--per-app-target', '5000'
]
subprocess.run(scrape_cmd, cwd=project_root, check=True)

## 2) Preprocessing + Pelabelan 3 Kelas
Pelabelan utama menggunakan score rating: 1-2 = negative, 3 = neutral, 4-5 = positive.

In [ ]:
prep_cmd = [
    python_exec,
    str(project_root / 'scripts' / 'prepare_dataset.py')
]
subprocess.run(prep_cmd, cwd=project_root, check=True)

In [ ]:
import pandas as pd
processed_path = project_root / 'data' / 'processed' / 'playstore_reviews_processed_latest.csv'
df = pd.read_csv(processed_path)
print('Processed rows:', len(df))
display(df['sentiment'].value_counts())
display(df[['brand', 'sentiment']].value_counts().reset_index(name='count').head(10))

## 3) Pelatihan 3 Skema Berbeda
Skema: 
1. SVM + TF-IDF + split 80/20
2. Logistic Regression + TF-IDF + split 70/30
3. BiLSTM + sequence tokenizer + split 80/20

In [ ]:
train_cmd = [
    python_exec,
    str(project_root / 'scripts' / 'train_experiments.py')
]
subprocess.run(train_cmd, cwd=project_root, check=True)

In [ ]:
metrics_path = project_root / 'reports' / 'metrics_experiments.csv'
metrics_df = pd.read_csv(metrics_path)
display(metrics_df)
print('Jumlah eksperimen dengan test accuracy >= 85%:', (metrics_df['test_accuracy'] >= 0.85).sum())
print('Jumlah eksperimen dengan train/test >= 92%:', ((metrics_df['train_accuracy'] >= 0.92) & (metrics_df['test_accuracy'] >= 0.92)).sum())

## 4) Inference (Output Kelas Kategorikal)
Cell ini menjadi bukti inferensi dengan output label `negative`, `neutral`, atau `positive`.

In [ ]:
infer_cmd = [
    python_exec,
    str(project_root / 'scripts' / 'run_inference.py'),
    '--text', 'Aplikasi bagus, driver cepat, saya puas',
    '--text', 'Aplikasinya sering error dan susah dipakai',
    '--text', 'Biasa saja, tidak terlalu bagus dan tidak buruk'
]
result = subprocess.run(infer_cmd, cwd=project_root, check=True, capture_output=True, text=True)
print(result.stdout)